In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROVA CON SEED

In [1]:
# =============================================================================
# CRI-XAI PROJECT — FASE 2 BIS: MICROSIMULAZIONE CON ISTRUZIONE CONDIZIONATA
# (comune + fascia d'età + genere, non più solo comune)
#
# IDENTICO a 02 tranne la Cella 5 (assegnazione istruzione) e piccoli
# adattamenti nella Cella 7 dove la Cella 5 viene richiamata.
# Output separati (_BIS) per non sovrascrivere la versione precedente.
# =============================================================================


# %% CELL 1 — Import e caricamento file comuni + POSAS (età singola)
# =============================================================================
# INVARIATA rispetto a 02
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PATH_COMUNI = './Materiale/dataset_random_forest_completo.csv'
PATH_POSAS = './Materiale/POSAS_2022_it_Comuni.csv'

df_comuni = pd.read_csv(PATH_COMUNI, sep=None, engine='python')
df_posas = pd.read_csv(PATH_POSAS, sep=';', encoding='utf-8-sig')

print(f"Comuni caricati: {len(df_comuni)}")
assert len(df_comuni) == 7904, f"Attenzione: attesi 7904 comuni, trovati {len(df_comuni)}"

df_posas_adulti = df_posas[(df_posas['Età'] != 999) & (df_posas['Età'] >= 15)].copy()

codici_comuni = set(df_comuni['Codice.comune'].unique())
codici_posas = set(df_posas_adulti['Codice.comune'].unique())
mancanti = codici_comuni - codici_posas
if mancanti:
    print(f"⚠️ {len(mancanti)} comuni presenti in df_comuni ma assenti in POSAS: {list(mancanti)[:10]}")
else:
    print(f"✅ Tutti i {len(codici_comuni)} comuni hanno una corrispondenza in POSAS (età singola).")

posas_by_comune = {code: g for code, g in df_posas_adulti.groupby('Codice.comune')}


# %% CELL 2 — Armonizzazione dei nomi delle regioni
# =============================================================================
# INVARIATA rispetto a 02
region_harmonization_map = {
    'Emilia Romagna': 'Emilia-Romagna',
    'Friuli Venezia Giulia': 'Friuli-Venezia Giulia',
    'Trentino Alto Adige(P.A.Bolzano)': 'Trentino-Alto Adige',
    'Trentino Alto Adige(P.A.Trento)': 'Trentino-Alto Adige',
    "Valle d'Aosta": 'Valle d’Aosta',
}

df_comuni['Region_Harmonized'] = df_comuni['Regione'].replace(region_harmonization_map)

valid_regions_training = [
    'Piemonte', 'Valle d’Aosta', 'Lombardia', 'Trentino-Alto Adige', 'Veneto',
    'Friuli-Venezia Giulia', 'Liguria', 'Emilia-Romagna', 'Toscana', 'Umbria',
    'Marche', 'Lazio', 'Abruzzo', 'Molise', 'Campania', 'Puglia', 'Basilicata',
    'Calabria', 'Sicilia', 'Sardegna'
]

unmatched = df_comuni[~df_comuni['Region_Harmonized'].isin(valid_regions_training)]
if len(unmatched) > 0:
    print("❌ COMUNI SENZA REGIONE ARMONIZZATA CORRETTAMENTE:")
    print(unmatched[['Comune', 'Regione', 'Region_Harmonized']].drop_duplicates())
    raise ValueError(f"{len(unmatched)} comuni non hanno trovato corrispondenza.")
else:
    print(f"✅ Tutti i {len(df_comuni)} comuni hanno una regione armonizzata correttamente.")


# %% CELL 3 — Preparazione dei marginali (reddito) + fasce età per r_i
# =============================================================================
# INVARIATA rispetto a 02
age_bracket_edges = [15, 25, 45, 65, 150]
age_bracket_labels = ['15-24', '25-44', '45-64', '65+']

income_brackets = [
    ('Reddito complessivo da 0 a 10000 euro - Frequenza',
     'Reddito complessivo da 0 a 10000 euro - Ammontare in euro', 0, 10_000),
    ('Reddito complessivo da 10000 a 15000 euro - Frequenza',
     'Reddito complessivo da 10000 a 15000 euro - Ammontare in euro', 10_000, 15_000),
    ('Reddito complessivo da 15000 a 26000 euro - Frequenza',
     'Reddito complessivo da 15000 a 26000 euro - Ammontare in euro', 15_000, 26_000),
    ('Reddito complessivo da 26000 a 55000 euro - Frequenza',
     'Reddito complessivo da 26000 a 55000 euro - Ammontare in euro', 26_000, 55_000),
    ('Reddito complessivo da 55000 a 75000 euro - Frequenza',
     'Reddito complessivo da 55000 a 75000 euro - Ammontare in euro', 55_000, 75_000),
    ('Reddito complessivo da 75000 a 120000 euro - Frequenza',
     'Reddito complessivo da 75000 a 120000 euro - Ammontare in euro', 75_000, 120_000),
    ('Reddito complessivo oltre 120000 euro - Frequenza',
     'Reddito complessivo oltre 120000 euro - Ammontare in euro', 120_000, 300_000),
]

freq_cols = [f for f, _, _, _ in income_brackets]
df_comuni['_check_sum_freq'] = df_comuni[freq_cols].sum(axis=1)
scarto_medio = (df_comuni['_check_sum_freq'] - df_comuni['Numero contribuenti']).abs().mean()
print(f"\nScarto medio |somma frequenze reddito - Numero contribuenti|: {scarto_medio:.1f}")


# %% CELL 4 — Generazione della popolazione sintetica per comune
# =============================================================================
# INVARIATA rispetto a 02
def genera_popolazione_sintetica(row, posas_by_comune, age_bracket_edges,
                                   age_bracket_labels, income_brackets,
                                   random_state=None):
    rng = np.random.default_rng(random_state)
    n_contribuenti = int(row['Numero contribuenti'])
    if n_contribuenti <= 0:
        return pd.DataFrame()

    codice = row['Codice.comune']
    if codice not in posas_by_comune:
        return pd.DataFrame()
    posas_comune = posas_by_comune[codice]

    eta_vals = np.concatenate([posas_comune['Età'].values, posas_comune['Età'].values])
    sesso_vals = np.array(['M'] * len(posas_comune) + ['F'] * len(posas_comune))
    pesi = np.concatenate([posas_comune['Totale.maschi'].values,
                            posas_comune['Totale.femmine'].values]).astype(float)

    if pesi.sum() <= 0:
        return pd.DataFrame()

    probs = pesi / pesi.sum()
    sample_idx = rng.choice(len(eta_vals), size=n_contribuenti, p=probs)
    ages = eta_vals[sample_idx]
    genders = sesso_vals[sample_idx]

    age_brackets_assigned = pd.cut(ages, bins=age_bracket_edges, labels=age_bracket_labels,
                                    right=False, include_lowest=True).astype(str)

    freqs = np.array([row[f] for f, _, _, _ in income_brackets], dtype=float)
    freqs = np.nan_to_num(freqs, nan=0.0)
    freqs = np.clip(freqs, 0, None)
    income_probs = freqs / freqs.sum() if freqs.sum() > 0 else None
    if income_probs is None:
        return pd.DataFrame()

    income_bracket_idx = rng.choice(len(income_brackets), size=n_contribuenti, p=income_probs)
    lower_bounds = np.array([lo for _, _, lo, _ in income_brackets])[income_bracket_idx]
    upper_bounds = np.array([hi for _, _, _, hi in income_brackets])[income_bracket_idx]
    incomes = rng.uniform(lower_bounds, upper_bounds)

    df_synth = pd.DataFrame({
        'Comune_Codice': codice,
        'Region': row['Region_Harmonized'],
        'Gender': genders,
        'Age': ages,
        'Age_Bracket': age_brackets_assigned,
        'Total_Gross_Income': incomes,
    })
    return df_synth


sample_comuni = df_comuni.sample(20, random_state=RANDOM_STATE)
synth_chunks_test = []
for i, (_, row) in enumerate(sample_comuni.iterrows()):
    df_synth = genera_popolazione_sintetica(
        row, posas_by_comune, age_bracket_edges, age_bracket_labels,
        income_brackets, random_state=RANDOM_STATE + i
    )
    synth_chunks_test.append(df_synth)

df_synth_test = pd.concat(synth_chunks_test, ignore_index=True)
print(f"\n✅ Test su 20 comuni: generati {len(df_synth_test)} individui sintetici")


# %% CELL 5 BIS — Assegnazione istruzione CONDIZIONATA a età e genere
# =============================================================================
# QUESTA È L'UNICA CELLA CAMBIATA rispetto a 02. Invece di una singola
# proporzione per comune, usa quota_istruzione_comuni_BIS.csv — una riga per
# comune x Fascia_Eta x Gender — e assegna a ciascun individuo sintetico la
# proporzione specifica per LA SUA combinazione età+genere, non la media
# del comune intero.
PATH_CENSIMENTO_BIS = './Materiale/quota_istruzione_comuni_BIS.csv'
df_censimento_bis = pd.read_csv(PATH_CENSIMENTO_BIS)

# Indicizziamo per accesso rapido: chiave (Codice.comune, Fascia_Eta, Gender)
censimento_bis_by_key = df_censimento_bis.set_index(
    ['Codice.comune', 'Fascia_Eta', 'Gender']
).to_dict('index')

# NOTA IMPORTANTE: le fasce d'età del file di Fabio (0-14, 15-24, 25-44,
# 45-64, 65+) includono "0-14" — ma la nostra popolazione sintetica parte
# da 15 anni (solo contribuenti). L'individuo sintetico più giovane possibile
# ricade quindi sempre in "15-24" o oltre, "0-14" semplicemente non verrà mai
# cercata — nessun problema, solo per chiarezza.

macro_names = ['0_Base', '1_Medio', '2_Alto']


def assegna_istruzione_censimento_bis(df_synth, censimento_bis_by_key,
                                       quota_fallback=None, random_state=None):
    """
    Campiona l'istruzione per ogni individuo sintetico secondo la proporzione
    VERA specifica per (comune, fascia d'età, genere) — non più una media
    unica per comune. Se manca la combinazione esatta (raro: comuni piccoli
    con zero osservazioni in quella cella), usa un fallback fornito
    (es. la proporzione media del comune, dal file 01 non-BIS).


    """
    rng = np.random.default_rng(random_state)
    df_synth = df_synth.copy()
    education_assegnata = np.empty(len(df_synth), dtype=object)

    # Raggruppiamo per combinazione (comune, fascia, genere) — tutte le righe
    # con la stessa combinazione condividono le stesse probabilità
    for (comune, fascia, genere), gruppo in df_synth.groupby(
        ['Comune_Codice', 'Age_Bracket', 'Gender']
    ):
        chiave = (comune, fascia, genere)
        if chiave in censimento_bis_by_key:
            riga = censimento_bis_by_key[chiave]
            quote = np.array([
                riga['quota_0_Base'], riga['quota_1_Medio'], riga['quota_2_Alto']
            ], dtype=float)
        elif quota_fallback is not None and comune in quota_fallback:
            # Fallback: proporzione media del comune (senza dettaglio età/genere)
            riga = quota_fallback[comune]
            quote = np.array([
                riga['quota_istruzione_0_Base'], riga['quota_istruzione_1_Medio'],
                riga['quota_istruzione_2_Alto']
            ], dtype=float)
        else:
            # Nessun dato disponibile nemmeno come fallback: quota nazionale media
            quote = np.array([0.213, 0.673, 0.115])

        quote = quote / quote.sum()
        idx_gruppo = df_synth.index[
            (df_synth['Comune_Codice'] == comune) &
            (df_synth['Age_Bracket'] == fascia) &
            (df_synth['Gender'] == genere)
        ]
        scelte = rng.choice(3, size=len(idx_gruppo), p=quote)
        for pos, idx_reale in enumerate(idx_gruppo):
            education_assegnata[df_synth.index.get_loc(idx_reale)] = macro_names[scelte[pos]]

    df_synth['Education_Macro'] = education_assegnata
    return df_synth


# --- Carichiamo anche il fallback (versione senza dettaglio, dal file 01) ---
PATH_CENSIMENTO_FALLBACK = './Materiale/quota_istruzione_comuni_CENSIMENTO.csv'
df_censimento_fallback = pd.read_csv(PATH_CENSIMENTO_FALLBACK)
censimento_fallback_by_comune = df_censimento_fallback.set_index('Codice.comune').to_dict('index')

# --- Test sui 20 comuni ---
df_synth_test = assegna_istruzione_censimento_bis(
    df_synth_test, censimento_bis_by_key, quota_fallback=censimento_fallback_by_comune,
    random_state=RANDOM_STATE
)

print("\n✅ Istruzione (condizionata età/genere) assegnata al test dei 20 comuni:")
print(df_synth_test['Education_Macro'].value_counts())
print(df_synth_test.head(10))

n_mancanti_test = df_synth_test['Education_Macro'].isna().sum()
print(f"\nIndividui senza istruzione assegnata (dovrebbe essere 0): {n_mancanti_test}")


# %% CELL 6 — Calcolo di r_i (Młodak) — INVARIATA rispetto a 02
# =============================================================================
def calcola_r_i(df_synth, p_income=0.10, k_mef=4, lam=0.5):
    df = df_synth.copy()
    combo_cols = ['Comune_Codice', 'Gender', 'Age_Bracket', 'Education_Macro']
    df['f_i'] = df.groupby(combo_cols)['Comune_Codice'].transform('count')

    w_list = np.empty(len(df), dtype=int)
    for comune_code, idx in df.groupby('Comune_Codice').groups.items():
        idx = np.array(idx)
        incomes = df.loc[idx, 'Total_Gross_Income'].values
        order = np.argsort(incomes)
        sorted_incomes = incomes[order]
        n = len(sorted_incomes)

        lower = sorted_incomes * (1 - p_income)
        upper = sorted_incomes * (1 + p_income)
        left = np.searchsorted(sorted_incomes, lower, side='left')
        right = np.searchsorted(sorted_incomes, upper, side='right')
        eta = (right - left) - 1

        w_sorted = (eta < k_mef).astype(int)
        w_orig_order = np.empty(n, dtype=int)
        w_orig_order[order] = w_sorted
        w_list[idx] = w_orig_order

    df['w_income'] = w_list
    df['r_i'] = lam * (1 / df['f_i']) + (1 - lam) * df['w_income']
    return df


df_synth_test = calcola_r_i(df_synth_test)
print("\n✅ r_i calcolato sul test dei 20 comuni (versione BIS):")
print(df_synth_test[['Comune_Codice', 'Gender', 'Age', 'Education_Macro',
                      'Total_Gross_Income', 'f_i', 'w_income', 'r_i']].head(10))
print("\nDistribuzione di r_i:")
print(df_synth_test['r_i'].describe())


# %% CELL 7 BIS — GENERAZIONE COMPLETA SU TUTTI I 7.904 COMUNI
# =============================================================================
# Identica a 02 nella struttura, ma chiama assegna_istruzione_censimento_bis
# invece della versione senza dettaglio. Output con suffisso _BIS.
import os

BATCH_SIZE = 200
CAMPIONE_PER_COMUNE = 150

PATH_QUOTE_OUTPUT_BIS = './Materiale/quota_istruzione_comuni_finale_BIS.csv'
PATH_CAMPIONE_OUTPUT_BIS = './Materiale/campione_individuale_r_i_BIS.csv'
PATH_CHECKPOINT_BIS = './Materiale/_checkpoint_microsimulazione_BIS.txt'

comuni_gia_fatti = 0
if os.path.exists(PATH_CHECKPOINT_BIS):
    with open(PATH_CHECKPOINT_BIS, 'r') as f:
        comuni_gia_fatti = int(f.read().strip())
    print(f"⏯️ Ripresa da checkpoint BIS: {comuni_gia_fatti} comuni già processati.")
else:
    print("▶️ Nessun checkpoint BIS trovato, parto da zero.")
    for p in [PATH_QUOTE_OUTPUT_BIS, PATH_CAMPIONE_OUTPUT_BIS]:
        if os.path.exists(p):
            os.remove(p)

df_comuni_da_fare = df_comuni.iloc[comuni_gia_fatti:].reset_index(drop=True)
n_batch_totali = (len(df_comuni_da_fare) // BATCH_SIZE) + 1

for batch_idx in range(n_batch_totali):
    batch = df_comuni_da_fare.iloc[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
    if len(batch) == 0:
        break

    quote_batch = []
    campioni_batch = []

    for i, (_, row) in enumerate(batch.iterrows()):
        seed_comune = RANDOM_STATE + comuni_gia_fatti + batch_idx * BATCH_SIZE + i
        df_synth = genera_popolazione_sintetica(
            row, posas_by_comune, age_bracket_edges, age_bracket_labels,
            income_brackets, random_state=seed_comune
        )
        if len(df_synth) == 0:
            continue

        df_synth = assegna_istruzione_censimento_bis(
            df_synth, censimento_bis_by_key, quota_fallback=censimento_fallback_by_comune,
            random_state=seed_comune
        )
        df_synth = calcola_r_i(df_synth)

        quote = df_synth['Education_Macro'].value_counts(normalize=True)
        quote_batch.append({
            'Codice.comune': row['Codice.comune'],
            'quota_istruzione_0_Base': quote.get('0_Base', 0.0),
            'quota_istruzione_1_Medio': quote.get('1_Medio', 0.0),
            'quota_istruzione_2_Alto': quote.get('2_Alto', 0.0),
            'n_individui_sintetici': len(df_synth),
        })

        n_campione = min(CAMPIONE_PER_COMUNE, len(df_synth))
        campioni_batch.append(df_synth.sample(n_campione, random_state=seed_comune))

    df_quote_batch = pd.DataFrame(quote_batch)
    header_quote = not os.path.exists(PATH_QUOTE_OUTPUT_BIS)
    df_quote_batch.to_csv(PATH_QUOTE_OUTPUT_BIS, mode='a', header=header_quote, index=False)

    if campioni_batch:
        df_campione_batch = pd.concat(campioni_batch, ignore_index=True)
        header_campione = not os.path.exists(PATH_CAMPIONE_OUTPUT_BIS)
        df_campione_batch.to_csv(PATH_CAMPIONE_OUTPUT_BIS, mode='a', header=header_campione, index=False)

    comuni_processati_ora = comuni_gia_fatti + min((batch_idx + 1) * BATCH_SIZE, len(df_comuni_da_fare))
    with open(PATH_CHECKPOINT_BIS, 'w') as f:
        f.write(str(comuni_processati_ora))

    print(f"Batch {batch_idx + 1}/{n_batch_totali} — "
          f"comuni totali processati: {comuni_processati_ora}/{len(df_comuni)}")

print(f"\n✅ COMPLETATO (BIS) — output finali:")
print(f"   Proporzioni per comune (aggregate da r_i BIS): {PATH_QUOTE_OUTPUT_BIS}")
print(f"   Campione individuale per R_i (BIS): {PATH_CAMPIONE_OUTPUT_BIS}")

if comuni_processati_ora >= len(df_comuni):
    os.remove(PATH_CHECKPOINT_BIS)
    print("   Checkpoint BIS rimosso: lavoro concluso con successo.")


# %% CELL 8 BIS — Controllo finale
# =============================================================================
df_quote_check_bis = pd.read_csv(PATH_QUOTE_OUTPUT_BIS)
print(f"Righe in quota_istruzione_comuni_finale_BIS.csv: {len(df_quote_check_bis)}")
print(f"Comuni ancora mancanti rispetto a 7.904: {7904 - len(df_quote_check_bis)}")

df_campione_check_bis = pd.read_csv(PATH_CAMPIONE_OUTPUT_BIS)
print(f"Righe nel campione individuale BIS: {len(df_campione_check_bis):,}")

Comuni caricati: 7904
✅ Tutti i 7904 comuni hanno una corrispondenza in POSAS (età singola).
✅ Tutti i 7904 comuni hanno una regione armonizzata correttamente.

Scarto medio |somma frequenze reddito - Numero contribuenti|: 130.3

✅ Test su 20 comuni: generati 63734 individui sintetici

✅ Istruzione (condizionata età/genere) assegnata al test dei 20 comuni:
Education_Macro
1_Medio    44756
0_Base     10458
2_Alto      8520
Name: count, dtype: int64
   Comune_Codice  Region Gender  Age Age_Bracket  Total_Gross_Income  \
0          29027  Veneto      F   58       45-64        22311.042100   
1          29027  Veneto      M   76         65+          336.099692   
2          29027  Veneto      F   68         65+        18031.985562   
3          29027  Veneto      F   50       45-64          942.298644   
4          29027  Veneto      M   35       25-44         1907.271573   
5          29027  Veneto      F   87         65+        22026.040042   
6          29027  Veneto      F   56       4